Setup

In [2]:
import os, subprocess
print(subprocess.run(["java", "-version"], capture_output=True, text=True).stderr)

import pyspark
print("pyspark", pyspark.__version__)

from pyspark.sql import SparkSession, functions as F
from pyspark.sql.window import Window
from pyspark.ml import Pipeline, PipelineModel
from pyspark.ml.feature import RegexTokenizer, StopWordsRemover, HashingTF, IDF
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.functions import vector_to_array
import pandas as pd

# Paths
MODEL_PATH = "ag_news_model"
STREAM_PATH = "posts.txt"    # export file from topic1
PRED_CSV = "predictions.csv"
HAND_CSV = "hand_labels.csv"

spark = (SparkSession.builder.appName("gp1-part2")
         .master("local[*]")
         .config("spark.driver.memory", "6g")
         .config("spark.driver.maxResultSize", "2g")
         .getOrCreate())
print("driver memory:", spark.conf.get("spark.driver.memory"))

openjdk version "21.0.12.1" 2026-08-18
OpenJDK Runtime Environment (build 21.0.12.1+1-1-24.04.4-Ubuntu)
OpenJDK 64-Bit Server VM (build 21.0.12.1+1-1-24.04.4-Ubuntu, mixed mode, sharing)

pyspark 4.0.5
driver memory: 6g


AG News load and class counts

In [3]:
from huggingface_hub import hf_hub_download, list_repo_files

try:
    path = hf_hub_download(repo_id="fancyzhx/ag_news", repo_type="dataset",
                           filename="data/train-00000-of-00001.parquet")
except Exception:
    print("Download failed. Files in the repo:")
    print(list_repo_files("fancyzhx/ag_news", repo_type="dataset"))
    raise

train = (spark.read.parquet(path)
         .select("text", F.col("label").cast("double").alias("label"))
         .dropna())

print("total articles:", train.count())
train.groupBy("label").count().orderBy("label").show()   # 0 World, 1 Sports, 2 Business, 3 Sci/Tech

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 18.6MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

total articles: 120000
+-----+-----+
|label|count|
+-----+-----+
|  0.0|30000|
|  1.0|30000|
|  2.0|30000|
|  3.0|30000|
+-----+-----+



pipeline, fit, and save

In [4]:
pipeline = Pipeline(stages=[
    RegexTokenizer(inputCol="text", outputCol="tokens", pattern="\\W+", toLowercase=True),
    StopWordsRemover(inputCol="tokens", outputCol="filtered"),
    HashingTF(inputCol="filtered", outputCol="tf", numFeatures=1 << 16),
    IDF(inputCol="tf", outputCol="features", minDocFreq=2),
    LogisticRegression(featuresCol="features", labelCol="label", maxIter=20),
])
train = train.repartition(8).cache()
model = pipeline.fit(train)
model.write().overwrite().save(MODEL_PATH)
print("model saved to", MODEL_PATH)

model saved to ag_news_model


streamed data load and cleaning

In [5]:
if not os.path.exists(STREAM_PATH):
    raise FileNotFoundError(
        f"{STREAM_PATH} not found. Upload posts.txt in the Files sidebar "
        "and set STREAM_PATH to its exact name.")

raw = spark.read.text(STREAM_PATH).select(F.col("value").alias("text"))
print("raw messages:", raw.count())
clean = (raw.select(F.trim(F.regexp_replace("text", "\\r", "")).alias("text"))
            .filter(F.col("text").isNotNull() & (F.col("text") != ""))
            .dropDuplicates(["text"]))
print("after removing empty/duplicate:", clean.count())

raw messages: 1000
after removing empty/duplicate: 998


batch predictions

In [6]:
model = PipelineModel.load(MODEL_PATH)
labels = ["World", "Sports", "Business", "Sci/Tech"]
mapping = F.create_map(*[x for i, l in enumerate(labels)
                         for x in (F.lit(float(i)), F.lit(l))])

pred = (model.transform(clean)
        .withColumn("predicted_class", mapping[F.col("prediction")])
        .withColumn("probability", F.array_max(vector_to_array("probability")))
        .select("text", "predicted_class", "probability")
        .cache())

pred.groupBy("predicted_class").count().orderBy("predicted_class").show()

w = Window.partitionBy("predicted_class").orderBy(F.rand(42))
(pred.withColumn("rn", F.row_number().over(w))
     .filter("rn <= 5")
     .select("predicted_class", "probability", "text")
     .orderBy("predicted_class")
     .show(40, truncate=100))

out = pred.toPandas()
out.to_csv(PRED_CSV, index=False)
print("saved", PRED_CSV, len(out), "rows")

+---------------+-----+
|predicted_class|count|
+---------------+-----+
|       Business|  404|
|       Sci/Tech|  323|
|         Sports|  154|
|          World|  117|
+---------------+-----+

+---------------+------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|predicted_class|       probability|                                                                                                                                                                  text|
+---------------+------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|       Business|0.7576051492784848|                                                                                                                                                           No! 

In [13]:
w = Window.partitionBy("predicted_class").orderBy(F.rand(42))
(pred.withColumn("rn", F.row_number().over(w))
     .filter("rn <= 5")
     .select("predicted_class", F.round("probability", 3).alias("probability"), "text")
     .orderBy("predicted_class")
     .show(40, truncate=60))

+---------------+-----------+---------------------------------------------------------------------------------------------------+
|predicted_class|probability|                                                                                               text|
+---------------+-----------+---------------------------------------------------------------------------------------------------+
|       Business|      0.758|                                                                                        No! Really?|
|       Business|      0.708|                                       #ReleaseTrumpEpsteinFiles #ReleaseTrumpEpsteinFiles #Rele...|
|       Business|      0.991|                                       The feedback questionnaire is basically a siren call with...|
|       Business|        1.0|                                       This flowing practice reconnects us with the natural rhyt...|
|       Business|      0.759|                                                             

sample for hand labeling

In [7]:
sample = out.sample(n=min(50, len(out)), random_state=42).reset_index(drop=True)
sample["my_label"] = ""
sample.to_csv(HAND_CSV, index=False)
print("saved", HAND_CSV)

saved hand_labels.csv


In [8]:
sample = pd.read_csv(HAND_CSV, keep_default_na=False)
for i, t in enumerate(sample["text"]):
    print(i, repr(t[:300]))

0 'REACHER Wood Grain Sunrise Alarm Clock and Sound Machine Nightlight Digital Dimmable Clock for Bedroom 26 Sleep Sounds White Noise Machine for Baby Adults Wake Up Light Alarm Clock for Kids #white噪音 #dimmable #sleepsounds #bedroom #reacheralarm'
1 'Nope...he can swing on over to the party of pedophiles any time he likes...'
2 'Sapporo Cityscape Hokkaido, Japan #Japan #Nippon'
3 '"King Arthur: Overture Z 628" by Henry Purcell / @englishconcert.bsky.social, conducted by Trevor Pinnock / EMI records sp. 40329'
4 'fgo辞めてしまった後にダイナマイト利休なるものが出て悔しくて泣いてる'
5 'novo meme:'
6 'The AI agent inside Google Maps just quietly leveled up.'
7 "Whether they're a gadget geek, a hands-on tinkerer, a gamer, or a nostalgia seeker, we've curated the perfect collection of gifts to delight every kind of tech lover."
8 'sometimes i just think about how fucking hot lillie from pokemon is'
9 'MoBoIさんの作品「white」 https://www.pixiv.net/artworks/152155866 #VOCALOID #初音ミク #HatsuneMiku #イラスト'
10 'continuem clonando o Lu

In [9]:
codes = "N N N N N N T N N N N N N N N N W T N N T T N N N N W N N B N N N N N N N W N N N N N N N N N N N W"

code_map = {"W": "World", "S": "Sports", "B": "Business", "T": "Sci/Tech", "N": "None"}
letters = codes.split()
assert len(letters) == 50, f"need 50 letters, got {len(letters)}"
assert all(c in code_map for c in letters), "only W, S, B, T, N allowed"

sample["my_label"] = [code_map[c] for c in letters]
sample.to_csv(HAND_CSV, index=False)
print(sample["my_label"].value_counts())

my_label
None        41
Sci/Tech     4
World        4
Business     1
Name: count, dtype: int64


accuracy

In [10]:
d = pd.read_csv(HAND_CSV, keep_default_na=False)
d["my_label"] = d["my_label"].str.strip()

valid = {"World", "Sports", "Business", "Sci/Tech", "None"}
bad = d[~d["my_label"].isin(valid)]
if len(bad):
    print("Fix these rows (invalid or blank my_label):")
    print(bad[["my_label"]])
else:
    fits = d[d["my_label"] != "None"]
    print("fit one of four classes:", len(fits), "| fit none:", (d["my_label"] == "None").sum())
    print("accuracy:", (fits["my_label"] == fits["predicted_class"]).mean())

fit one of four classes: 9 | fit none: 41
accuracy: 0.4444444444444444


In [12]:
d = pd.read_csv(HAND_CSV, keep_default_na=False)
fits = d[d["my_label"] != "None"]
print(fits[["my_label", "predicted_class", "probability"]].to_string())
wrong = fits[fits["my_label"] != fits["predicted_class"]]
for _, r in wrong.iterrows():
    print("\nMine:", r["my_label"], "| Model:", r["predicted_class"], round(float(r["probability"]), 3))
    print(r["text"][:300])

    my_label predicted_class  probability
6   Sci/Tech           World     0.999111
16     World        Business     0.999441
17  Sci/Tech        Business     1.000000
20  Sci/Tech        Sci/Tech     0.932391
21  Sci/Tech        Sci/Tech     1.000000
26     World           World     0.776695
29  Business        Business     0.999689
37     World        Sci/Tech     0.822397
49     World        Sci/Tech     0.999939

Mine: Sci/Tech | Model: World 0.999
The AI agent inside Google Maps just quietly leveled up.

Mine: World | Model: Business 0.999
we wanna be a state no y we want to retain the power to hurt you and we like that you can’t do anything about it

Mine: Sci/Tech | Model: Business 1.0
📻 BAND CONDITIONS (2/5) 160m: 🟢 Good 80m: 🟢 Good 40m: 🟢 Good 30m: 🟢 Good 20m: 🟢 Good 17m: 🟢 Good 15m: 🟡 Fair 12m: 🟡 Fair 10m: 🟡 Fair 6m: 🔴 Closed 🎯 Best Now: 80m, 40m, 30m, 20m Based on MUF=30.0MHz #SolarStormScout #HamRadio

Mine: World | Model: Sci/Tech 0.822
Also it doesn't suddenly get fair to

In [11]:
import os
print(os.path.exists(MODEL_PATH), MODEL_PATH)
!zip -rq ag_news_model.zip {MODEL_PATH}
from google.colab import files
files.download("ag_news_model.zip")

True ag_news_model


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>